# TextCNN — Single-Task Models
Trains two independent TextCNN models, one per target label.  
Each model only sees its own label during training (no shared backbone).  
Compare with `Sprint_2.ipynb` which trains a single multi-task model on both labels simultaneously.  
Train/dev only — test set not touched.

Had help from Claude on implementation.

In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch
import cnn_baseline as cnn

from config import DATA_DIR, FASTTEXT_PATH, TARGETS
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")

Device: mps


In [8]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


In [9]:
# Build vocab and embeddings once — shared across both models
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
print(f"Vocab size: {len(vocab):,} | Embedding matrix: {tuple(embed_matrix.shape)}")

Loading FastText vectors from /Users/jennifer/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Vocab size: 4,091 | Embedding matrix: (4091, 300)


In [10]:
# Train one independent model per target
# Each model only optimises for its own label — no shared backbone
results = {}
models  = {}

train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"Training: {target}")
    print(f"{'='*55}")
    model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
                            train_targets=[target])
    models[target]               = model
    results[f"TextCNN — {target}"] = cnn.predict(model, dev_loader, DEVICE, target=target)


Training: opinion_label


TypeError: train_model() got an unexpected keyword argument 'train_targets'

In [ ]:
# Metrics comparison table
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (class 0)": m["f1_class0"], "F1 (class 1)": m["f1_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (class 0),F1 (class 1),AUC-ROC
Model,,,,,
TextCNN — opinion_label,0.7050,0.7037,0.7230,0.6845,0.7754
TextCNN — misinformation_label,0.9000,0.8684,0.9329,0.8039,0.9584


In [ ]:
# Per-model detail
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN — opinion_label
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    77      40
  true=1 (opinion):   19      64

              precision    recall  f1-score   support

 not-opinion       0.80      0.66      0.72       117
     opinion       0.62      0.77      0.68        83

    accuracy                           0.70       200
   macro avg       0.71      0.71      0.70       200
weighted avg       0.72      0.70      0.71       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTenni